# Hard-adversarial ranking check — Wikidata

For each held-out anchor, does the model score `cos(anchor, positive)` HIGHER than `cos(anchor, negative)`, where the positive is answer-equivalent but lexically divergent, and the negative is lexically similar but answer-different?

A lexical-only encoder should get this backwards; a result-aware encoder (SparqlCE) should get it right. Run once on all rows, then again filtered to `meets_strict_bar: true` only — that's the headline number (see `datasets/README.md` for how few rows actually clear both thresholds).

In [ ]:
import sys, os
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

_RQ1_DIR = Path.cwd().parent
if str(_RQ1_DIR) not in sys.path:
    sys.path.insert(0, str(_RQ1_DIR))

from evaluate_hard_adversarial import load_hard_testset, evaluate_all
print(f"Project root: {_PROJECT_ROOT}")

TEST_DATA_PATH = os.path.join(_PROJECT_ROOT, "datasets", "wikidata_testset_RQ1_hard.json")
CHECKPOINT_PATH = os.path.join(_PROJECT_ROOT, "sparqlce", "models", "wikidata_model_mrr_0.3238.pt")
E5_FT_PATH = os.path.join(_PROJECT_ROOT, "checkpoints", "e5_square_wikidata", "best_model")
BATCH = 32


## Full set (all rows, includes rows below the strict bar — noisier, larger-N check)

In [ ]:
rows_all = load_hard_testset(TEST_DATA_PATH, strict_only=False)
results_all = evaluate_all(rows_all, CHECKPOINT_PATH, e5_ft_path=E5_FT_PATH, batch_size=BATCH, label="(full) ")


## Strict-bar rows only (the headline number — see `meets_strict_bar` in the README)

In [ ]:
rows_strict = load_hard_testset(TEST_DATA_PATH, strict_only=True)
results_strict = evaluate_all(rows_strict, CHECKPOINT_PATH, e5_ft_path=E5_FT_PATH, batch_size=BATCH, label="(strict-bar) ")
